# Phase 3 — Fine-tune Embedding Models

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Instructions:**
1. Run **Cell 1** (install) — it will automatically restart the runtime
2. After restart, run **Cells 2 → 5** in order
3. Cell 2 will pop up a file picker — upload `train.jsonl` and `eval.jsonl` from your local `data/` folder

In [ ]:
# Cell 1 — Install dependencies + auto-restart runtime
# Run this cell ONCE. It will restart the runtime automatically.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'sentence-transformers>=3.0.0', 'datasets>=2.19.0', 'accelerate>=1.1.0'])

print('Packages installed. Restarting runtime...')
import os
os.kill(os.getpid(), 9)  # force clean restart so new packages load correctly

In [ ]:
# Cell 2 — Upload train.jsonl and eval.jsonl
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print('Select train.jsonl and eval.jsonl from your local data/ folder')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  saved → {dest} ({len(content)/1024:.1f} KB)')

assert os.path.exists('data/train.jsonl'), 'Missing train.jsonl'
assert os.path.exists('data/eval.jsonl'),  'Missing eval.jsonl'
print('Both files ready.')

In [ ]:
# Cell 3 — Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'Device: {device}')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 4 — Fine-tune all 3 models
import json, re
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

MODELS = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'BAAI/bge-small-en-v1.5',
    'BAAI/bge-base-en-v1.5',
]
EPOCHS       = 3
BATCH_SIZE   = 32    # T4 16GB VRAM can handle 32
LR           = 2e-5
WARMUP_RATIO = 0.1

def load_jsonl(path):
    return [json.loads(l) for l in open(path)]

def build_dataset(triplets):
    return Dataset.from_dict({
        'anchor':   [t['query']    for t in triplets],
        'positive': [t['positive'] for t in triplets],
    })

def model_slug(name):
    return re.sub(r'[^\w]', '-', name.split('/')[-1]).lower()

train_triplets = load_jsonl('data/train.jsonl')
eval_triplets  = load_jsonl('data/eval.jsonl')
print(f'Train: {len(train_triplets)}  |  Eval: {len(eval_triplets)}')

train_dataset = build_dataset(train_triplets)
eval_dataset  = build_dataset(eval_triplets)

for model_name in MODELS:
    slug        = model_slug(model_name)
    output_path = f'finetuned/{slug}-finetuned'
    print(f'\n{"="*60}')
    print(f'Fine-tuning : {model_name}')
    print(f'Output      : {output_path}')
    print(f'{"="*60}')

    model = SentenceTransformer(model_name, device=device)
    loss  = losses.MultipleNegativesRankingLoss(model)
    warmup_steps = int(WARMUP_RATIO * len(train_dataset) / BATCH_SIZE * EPOCHS)

    args = SentenceTransformerTrainingArguments(
        output_dir=output_path,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        warmup_steps=warmup_steps,
        fp16=True,    # T4 supports fp16
        bf16=False,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        logging_steps=10,
        save_total_limit=1,
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=loss,
    )

    trainer.train()
    model.save(output_path)
    print(f'Saved → {output_path}')

print('\nAll models fine-tuned. Run Cell 5 to download.')

In [ ]:
# Cell 5 — Zip and download all checkpoints
import shutil
from google.colab import files

shutil.make_archive('finetuned_models', 'zip', 'finetuned')
size_mb = os.path.getsize('finetuned_models.zip') / 1e6
print(f'finetuned_models.zip  ({size_mb:.0f} MB)')
print('Downloading...')
files.download('finetuned_models.zip')
print('Done!')

## After downloading

```bash
# Unzip into your project root
unzip finetuned_models.zip -d finetuned/

# Verify
ls finetuned/
# all-minilm-l6-v2-finetuned/
# bge-small-en-v1-5-finetuned/
# bge-base-en-v1-5-finetuned/

# Run full benchmark — all 8 models now active
python -m scripts.run_benchmark
```